In [15]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
import shutil
import torch.nn.functional as F
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
import math

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

In [16]:
# =========================================================
# 1) GaussianKernelLayer
#    Entrada: x de forma (N, C, T, F)
#    Salida:  K de forma (N, C, C, F)   [igual a tu TF]
# =========================================================
class GaussianKernelLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, sigma):
        """
        x: tensor (N, C, T, F)
        sigma: escalar, float o tensor compatible
        retorna:
            gaussian_kernel: (N, C, C, F)
        """
        if not torch.is_tensor(sigma):
            sigma = torch.tensor(sigma, dtype=x.dtype, device=x.device)
        else:
            sigma = sigma.to(dtype=x.dtype, device=x.device)

        N, C, T, Fdim = x.shape

        # (N, C, T, F) -> (N, F, C, T)
        x = x.permute(0, 3, 1, 2)

        # (N, F, C, T) -> (N*F, C, T)
        x_reshaped = x.reshape(N * Fdim, C, T)

        # Distancias cuadradas por pares entre canales
        diff = x_reshaped.unsqueeze(2) - x_reshaped.unsqueeze(1)   # (N*F, C, C, T)
        diff2 = diff.pow(2)
        pairwise_distances_squared = diff2.sum(dim=-1)             # (N*F, C, C)

        # (N*F, C, C) -> (N, F, C, C) -> (N, C, C, F)
        pairwise_distances_squared = pairwise_distances_squared.reshape(N, Fdim, C, C)
        pairwise_distances_squared = pairwise_distances_squared.permute(0, 2, 3, 1)

        gaussian_kernel = torch.exp(-pairwise_distances_squared / (2.0 * sigma.pow(2)))
        return gaussian_kernel


# =========================================================
# 2) RenyiEntropyLayer
#    Entrada: K de forma (B, F, C, C)
#    Salida:  H de forma (B, F)
# =========================================================
class RenyiEntropyLayer(nn.Module):
    def __init__(self, alpha=2, eps=1e-8, normalize_by_logc=False):
        super().__init__()
        self.alpha = alpha
        self.eps = eps
        self.normalize_by_logc = normalize_by_logc

    def forward(self, K):
        """
        K: (B, F, C, C)
        retorna:
            H: (B, F)
        """
        K = K.float()

        # Normalización por traza
        trace = torch.diagonal(K, dim1=-2, dim2=-1).sum(dim=-1)    # (B, F)
        trace = torch.clamp(trace, min=self.eps)
        A = K / trace.unsqueeze(-1).unsqueeze(-1)                  # (B, F, C, C)

        if self.alpha == 2:
            AAT = A @ A.transpose(-1, -2)                          # (B, F, C, C)
            tr_AAT = torch.diagonal(AAT, dim1=-2, dim2=-1).sum(dim=-1)  # (B, F)
            H = -torch.log(torch.clamp(tr_AAT, min=self.eps))
        else:
            eigvals = torch.linalg.eigvalsh(A)                     # (B, F, C)
            eigvals = torch.clamp(eigvals.real, min=self.eps)
            H = torch.log(torch.sum(eigvals.pow(self.alpha), dim=-1)) / (1.0 - self.alpha)

        if self.normalize_by_logc:
            C = K.shape[-1]
            H = H / torch.log(torch.tensor(max(C, 2.0), dtype=H.dtype, device=H.device))

        return H


# =========================================================
# 3) RenyiEntropyRegularizer
#    Recibe salida de entropía (B, 1) o (B, F)
#    Devuelve vector (B,)
# =========================================================
class RenyiEntropyRegularizer(nn.Module):
    def __init__(self, mode="min"):
        super().__init__()
        if mode not in ["min", "max"]:
            raise ValueError("mode debe ser 'min' o 'max'")
        self.mode = mode

    def forward(self, y_pred):
        """
        y_pred: (B, 1) o (B, F)
        retorna:
            loss_per_sample: (B,)
        """
        H = y_pred.float().mean(dim=-1)   # (B,)

        if self.mode == "min":
            return H
        else:
            return -H


# =========================================================
# 4) inception_block equivalente
# =========================================================
def inception_block(x, kernel_sigma, gaussian_layer=None):
    """
    Versión de un solo kernel.
    x: (N, C, T, F)
    retorna: (N, C, C, F)
    """
    if gaussian_layer is None:
        gaussian_layer = GaussianKernelLayer()
    kernel = gaussian_layer(x, kernel_sigma)
    return kernel


# =========================================================
# 5) NormalizedBinaryCrossentropy
#    Asume y_pred probabilidades en [0,1], shape (B, 2)
#    Devuelve loss por muestra: (B,)
# =========================================================
class NormalizedBinaryCrossentropy(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def _bce_per_sample(self, y_true, y_pred):
        """
        Replica el comportamiento de tf.keras.losses.binary_crossentropy
        promediando en la última dimensión.
        """
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)
        bce = -(y_true * torch.log(y_pred) + (1.0 - y_true) * torch.log(1.0 - y_pred))
        return bce.mean(dim=-1)  # (B,)

    def forward(self, y_true, y_pred):
        """
        y_true: (B, 2)
        y_pred: (B, 2)
        retorna:
            cce_norm: (B,)
        """
        batch_size = y_pred.shape[0]
        device = y_pred.device
        dtype = y_pred.dtype

        cce = self._bce_per_sample(y_true, y_pred)

        left = torch.tensor([1.0, 0.0], device=device, dtype=dtype).unsqueeze(0).repeat(batch_size, 1)
        right = torch.tensor([0.0, 1.0], device=device, dtype=dtype).unsqueeze(0).repeat(batch_size, 1)

        cce_left = self._bce_per_sample(left, y_pred)
        cce_right = self._bce_per_sample(right, y_pred)

        cce_norm = cce / (cce_left + cce_right + self.eps)
        return cce_norm

class InspectableMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0, bias=True):
        super().__init__()

        if num_heads < 1:
            raise ValueError("num_heads debe ser >= 1")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.key_dim = max(1, embed_dim // num_heads)
        self.value_dim = self.key_dim
        self.inner_dim = self.num_heads * self.key_dim
        self.scale = math.sqrt(self.key_dim)

        self.q_proj = nn.Linear(embed_dim, self.inner_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, self.inner_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, self.inner_dim, bias=bias)
        self.out_proj = nn.Linear(self.inner_dim, embed_dim, bias=bias)

        self.dropout = nn.Dropout(dropout)
        self._last_attention_scores = None

    def forward(self, query, key=None, value=None, attention_mask=None):
        """
        query, key, value: (B, T, D)
        attention_mask: opcional, broadcastable a (B, H, Tq, Tk)
                        True = permitido, False = enmascarado
        """
        if key is None:
            key = query
        if value is None:
            value = query

        B, Tq, _ = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(B, Tq, self.num_heads, self.key_dim).transpose(1, 2)
        k = self.k_proj(key).view(B, Tk, self.num_heads, self.key_dim).transpose(1, 2)
        v = self.v_proj(value).view(B, Tk, self.num_heads, self.value_dim).transpose(1, 2)

        # scores: (B, H, Tq, Tk)
        scores = torch.matmul(q, k.transpose(-1, -2)) / self.scale

        if attention_mask is not None:
            if attention_mask.dtype != torch.bool:
                attention_mask = attention_mask.bool()
            scores = scores.masked_fill(~attention_mask, float("-inf"))

        attention_scores = torch.softmax(scores, dim=-1)
        attention_scores = self.dropout(attention_scores)

        out = torch.matmul(attention_scores, v)  # (B, H, Tq, d)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.inner_dim)
        out = self.out_proj(out)  # (B, Tq, D)

        self._last_attention_scores = attention_scores
        return out, attention_scores

    def get_projection_weights(self):
        # Igual idea que TF: devolver Q, K, V y O por separado
        return {
            "query": self.q_proj.weight.detach().cpu().numpy(),
            "key": self.k_proj.weight.detach().cpu().numpy(),
            "value": self.v_proj.weight.detach().cpu().numpy(),
            "output": self.out_proj.weight.detach().cpu().numpy(),
        }

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado aún los attention scores. Haz un forward pass primero."
            )
        return self._last_attention_scores



class InspectableTransformerEncoder(nn.Module):
    def __init__(
        self,
        embed_dim,
        num_heads,
        intermediate_dim,
        dropout=0.0,
        ff_activation=F.relu,   # <- más fiel al TransformerEncoder base de TF
    ):
        super().__init__()

        self.num_heads = num_heads
        self.dropout_rate = dropout
        self.intermediate_dim = intermediate_dim
        self.ff_activation = ff_activation

        self.self_attention = InspectableMultiHeadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

        # En tu encoder TF custom estas dos internas sí van con 1e-6
        self.self_attention_layer_norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.self_attention_dropout = nn.Dropout(dropout)

        self.feedforward_intermediate_dense = nn.Linear(embed_dim, intermediate_dim)
        self.feedforward_output_dense = nn.Linear(intermediate_dim, embed_dim)
        self.feedforward_dropout = nn.Dropout(dropout)
        self.feedforward_layer_norm = nn.LayerNorm(embed_dim, eps=1e-6)

        self._last_attention_scores = None

    def forward(self, inputs, padding_mask=None):
        attention_output, attention_scores = self.self_attention(
            query=inputs,
            key=inputs,
            value=inputs,
            attention_mask=padding_mask,
        )

        self._last_attention_scores = attention_scores

        attention_output = self.self_attention_dropout(attention_output)
        attention_output = self.self_attention_layer_norm(inputs + attention_output)

        ff_output = self.feedforward_intermediate_dense(attention_output)
        ff_output = self.ff_activation(ff_output)
        ff_output = self.feedforward_output_dense(ff_output)
        ff_output = self.feedforward_dropout(ff_output)

        output = self.feedforward_layer_norm(attention_output + ff_output)
        return output

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado attention scores todavía. Haz un forward pass primero."
            )
        return self._last_attention_scores

    def get_attention_weights(self):
        return self.self_attention.get_projection_weights()

# =========================================================
# 8) DynamicSchedule equivalente a callback de Keras
# =========================================================
class DynamicSchedule:
    def __init__(self, total_epochs, delta=10):
        self.total_epochs = total_epochs
        self.delta = delta
        self.lambda_val = 0.0

    def get_lambda(self, epoch):
        p = epoch / self.total_epochs
        return 2 * (1 - np.exp(-self.delta * p)) / (1 + np.exp(-self.delta * p))

    def on_epoch_begin(self, epoch, loss_weights):
        """
        loss_weights: dict, por ejemplo:
            {
                "out_activation": 1.0,
                "kernel_entropy": 0.0
            }
        """
        self.lambda_val = self.get_lambda(epoch)
        loss_weights["out_activation"] = 1.0
        loss_weights["kernel_entropy"] = self.lambda_val
        return loss_weights

In [17]:
class TGARNet(nn.Module):
    def __init__(
        self,
        nb_classes=2,
        Chans=19,
        Samples=512,
        norm_rate=0.25,
        alpha=2,
        num_heads=3,
        intermediate_dim=128,
        filters=3,
        dropoutRate=0.3,
        kernel_sigma=1.0,
        ff_activation=F.relu,   # <- corregido
        outer_ln_eps=1e-3,      # <- más fiel a LayerNormalization() de Keras
        bn_eps=1e-3,            # <- más fiel a BatchNormalization() de Keras
        bn_momentum=0.01,       # <- para aproximar el momentum=0.99 de Keras
        return_logits=False,
    ):
        super().__init__()

        self.nb_classes = nb_classes
        self.Chans = Chans
        self.Samples = Samples
        self.norm_rate = norm_rate
        self.alpha = alpha
        self.num_heads = num_heads
        self.intermediate_dim = intermediate_dim
        self.filters = filters
        self.dropoutRate = dropoutRate
        self.kernel_sigma = kernel_sigma
        self.return_logits = return_logits

        # 1) y 2) Transformer sobre (B, Samples, Chans)
        self.pre_ln = nn.LayerNorm(Chans, eps=outer_ln_eps)

        self.transformer_encoder = InspectableTransformerEncoder(
            embed_dim=Chans,
            num_heads=num_heads,
            intermediate_dim=intermediate_dim,
            dropout=dropoutRate,
            ff_activation=ff_activation,
        )

        # 4) Normalización después del Transformer
        self.post_ln = nn.LayerNorm(Chans, eps=outer_ln_eps)

        # 6) Single kernel block
        self.gaussian_layer = GaussianKernelLayer()

        # 7) Rényi entropy
        self.renyi_entropy = RenyiEntropyLayer(
            alpha=alpha,
            normalize_by_logc=True
        )

        # 8) Extra convolutional stack
        self.conv1 = nn.Conv2d(1, filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(filters, eps=bn_eps, momentum=bn_momentum)

        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(filters, eps=bn_eps, momentum=bn_momentum)

        self.dropout = nn.Dropout(dropoutRate)
        self.classifier = nn.Linear(filters * Chans * Chans, nb_classes)

    def forward(self, x):
        """
        x: (B, Chans, Samples)

        Devuelve lo mismo que el modelo TF:
            {
              'out_activation': probs,
              'kernel_entropy': kernel_entropy
            }

        Si return_logits=True, añade también 'logits'.
        """

        # 1) Reorganize data for Transformer (Samples, Chans)
        x = x.permute(0, 2, 1)  # (B, Samples, Chans)

        # 2) LayerNorm antes del Transformer
        x = self.pre_ln(x)

        # 3) Transformer
        x = self.transformer_encoder(x)

        # 4) LayerNorm después del Transformer
        x = self.post_ln(x)

        # 5) Restore original shape (Chans, Samples, 1)
        x = x.permute(0, 2, 1).unsqueeze(-1)  # (B, Chans, Samples, 1)

        # 6) Single kernel block
        kernel_out = self.gaussian_layer(x, self.kernel_sigma)  # (B, C, C, 1)

        # 7) Single-kernel Rényi entropy
        kernel_out_T = kernel_out.permute(0, 3, 1, 2)           # (B, 1, C, C)
        kernel_entropy = self.renyi_entropy(kernel_out_T)       # (B, 1)

        # 8) Extra convolutional stack
        final_conv = F.selu(self.conv1(kernel_out_T))
        final_conv = self.bn1(final_conv)

        final_conv = F.selu(self.conv2(final_conv))
        final_conv = self.bn2(final_conv)

        flat = torch.flatten(final_conv, start_dim=1)
        drop = self.dropout(flat)

        logits = self.classifier(drop)
        probs = torch.softmax(logits, dim=1)

        out = {
            "out_activation": probs,
            "kernel_entropy": kernel_entropy,
        }

        if self.return_logits:
            out["logits"] = logits

        return out


@torch.no_grad()
def apply_max_norm_to_linear(linear_layer, max_value=0.25, eps=1e-8):
    """
    Para nn.Linear:
        weight.shape = (out_features, in_features)

    Restringe la norma L2 de cada neurona de salida.
    """
    W = linear_layer.weight.data
    norms = W.norm(2, dim=1, keepdim=True).clamp_min(eps)
    desired = norms.clamp(max=max_value)
    W.mul_(desired / norms)

In [18]:
# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# WRAPPER PARA QUE SUMMARY VEA SOLO LA SALIDA PRINCIPAL
# =========================================================
class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        return out["out_activation"]   # (B, nb_classes)

# =========================================================
# CREAR MODELO TGARNet
# =========================================================
model = TGARNet(
    nb_classes=2,
    Chans=19,
    Samples=512,
    norm_rate=0.25,
    alpha=2,
    num_heads=3,
    intermediate_dim=128,
    filters=3,
    dropoutRate=0.3,
    kernel_sigma=1.0,
).to(device)

wrapped_model = SummaryWrapper(model).to(device)

# =========================================================
# SUMMARY TIPO TENSORFLOW
# =========================================================
summary(
    wrapped_model,
    input_size=(1, 19, 512),   # (batch, chans, samples)
    depth=4,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    device=device,
)

Using device: cuda


Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                                     [1, 19, 512]              [1, 2]                    --                        True
├─TGARNet: 1-1                                     [1, 19, 512]              [1, 1]                    --                        True
│    └─LayerNorm: 2-1                              [1, 512, 19]              [1, 512, 19]              38                        True
│    └─InspectableTransformerEncoder: 2-2          [1, 512, 19]              [1, 512, 19]              --                        True
│    │    └─InspectableMultiHeadAttention: 3-1     --                        [1, 512, 19]              --                        True
│    │    │    └─Linear: 4-1                       [1, 512, 19]              [1, 512, 18]              360                       True
│    │    │    └─Linear: 4-2                       [1, 51

In [19]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).
        labels (array): Etiquetas por sujeto en formato 0/1.

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas 0/1 por segmento
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos, dtype=np.float32), np.array(y, dtype=np.int64), sbjs, window_ids


def get_segmented_data():
    """
    Carga la base de datos y devuelve los segmentos listos para T-GARNet.
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_TDAH)
        if archivo.endswith(".mat")
    ])

    sujetos_control = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_control)
        if archivo.endswith(".mat")
    ])

    # =========================================================
    # ELIMINAR EXPLÍCITAMENTE EL SUJETO QUE NO ESTÁ EN folds.pkl
    # =========================================================
    sujeto_a_eliminar = "v36p"

    if sujeto_a_eliminar in sujetos_TDAH:
        sujetos_TDAH.remove(sujeto_a_eliminar)
        print(f"Sujeto eliminado explícitamente: {sujeto_a_eliminar}")
    else:
        print(f"ADVERTENCIA: {sujeto_a_eliminar} no está en sujetos_TDAH")

    print("Sujetos TDAH encontrados:", len(sujetos_TDAH))
    print("Sujetos Control encontrados:", len(sujetos_control))
    print("Sujetos totales encontrados:", len(sujetos_TDAH) + len(sujetos_control))

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah

    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    y = np.eye(2, dtype=np.float32)[y]

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [20]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).
        labels (array): Etiquetas por sujeto en formato 0/1.

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas 0/1 por segmento
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos, dtype=np.float32), np.array(y, dtype=np.int64), sbjs, window_ids


def get_segmented_data():
    """
    Carga la base de datos y devuelve los segmentos listos para T-GARNet.
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = [archivo[:-4] for archivo in os.listdir(ruta_carpeta_TDAH) if archivo.endswith('.mat')]
    sujetos_TDAH.pop()
    sujetos_control = [archivo[:-4] for archivo in os.listdir(ruta_carpeta_control) if archivo.endswith('.mat')]

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah
    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    # one-hot para T-GARNet: 0 -> [1,0], 1 -> [0,1]
    y = np.eye(2, dtype=np.float32)[y]

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [21]:
def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


class TGARNetCombinedLoss(nn.Module):
    """
    Réplica práctica de la lógica de compile() de Keras para TGARNet:
      total_loss =
          w_out * NormalizedBinaryCrossentropy(out_activation)
        + w_ent * RenyiEntropyRegularizer(kernel_entropy)

    `loss_weights` se comparte con DynamicSchedule para que cambie
    automáticamente el peso de kernel_entropy por época.
    """
    def __init__(self, loss_weights):
        super().__init__()
        self.loss_weights = loss_weights
        self.loss_out = NormalizedBinaryCrossentropy()
        self.loss_entropy = RenyiEntropyRegularizer(mode="min")

    def forward(self, outputs, y_true):
        loss_out = self.loss_out(y_true, outputs["out_activation"]).mean()
        loss_entropy = self.loss_entropy(outputs["kernel_entropy"]).mean()

        total_loss = (
            self.loss_weights["out_activation"] * loss_out
            + self.loss_weights["kernel_entropy"] * loss_entropy
        )
        return total_loss


def suggest_model_args(trial, model_name, base_model_args):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Este bloque limpio solo soporta model_name='TGARNet'.")

    return {
        **base_model_args,
        "filters": trial.suggest_int("filters", 2, 8),
        "kernel_sigma": trial.suggest_float("kernel_sigma", 1.0, 20.0),
        "num_heads": trial.suggest_int("num_heads", 1, 5),
        "intermediate_dim": trial.suggest_categorical("intermediate_dim", [16, 32, 64, 128]),
    }


def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Este bloque limpio solo soporta model_name='TGARNet'.")

    set_seed()

    return TGARNet(
        nb_classes=kwargs["nb_classes"],
        Chans=kwargs["Chans"],
        Samples=kwargs["Samples"],
        norm_rate=kwargs.get("norm_rate", 0.25),
        alpha=kwargs.get("alpha", 2),
        num_heads=kwargs["num_heads"],
        intermediate_dim=kwargs["intermediate_dim"],
        filters=kwargs["filters"],
        dropoutRate=kwargs.get("dropoutRate", 0.3),
        kernel_sigma=kwargs["kernel_sigma"],
        return_logits=False,
    )


def suggest_compile_args(trial, model_name, base_compile_args=None):
    if base_compile_args is None:
        base_compile_args = {}

    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Este bloque limpio solo soporta model_name='TGARNet'.")

    return {
        **base_compile_args,
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True),
        "schedule_delta": trial.suggest_float("schedule_delta", 1.0, 20.0),
    }


def build_compile_config(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Este bloque limpio solo soporta model_name='TGARNet'.")

    # Igual idea que en Keras: pesos iniciales 0.5 / 0.5
    # DynamicSchedule irá cambiando kernel_entropy durante el entrenamiento
    loss_weights = {
        "out_activation": 0.5,
        "kernel_entropy": 0.5,
    }

    compile_args = {
        "optimizer_name": "adam",
        "optimizer_params": {
            "lr": kwargs["learning_rate"],
        },
        "scheduler_name": None,
        "scheduler_params": {},
        "loss_fn": TGARNetCombinedLoss(loss_weights=loss_weights),
        "loss_weights": loss_weights,
    }

    callbacks = [
        DynamicSchedule(
            total_epochs=kwargs.get("total_epochs", kwargs.get("epochs", 100)),
            delta=kwargs["schedule_delta"],
        )
    ]

    return compile_args, callbacks

In [22]:
def ensure_onehot_labels(y):
    y = np.asarray(y)

    # ya viene one-hot [N, 2]
    if y.ndim == 2 and y.shape[1] == 2:
        return y.astype(np.float32)

    # caso [N]
    if y.ndim == 1:
        y = y.astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    # caso [N, 1]
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.squeeze(1).astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    raise ValueError(f"Formato de y no soportado para one-hot: shape={y.shape}")


def onehot_to_binary_labels(y):
    y = np.asarray(y)

    if y.ndim == 1:
        return y.astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1).astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 2:
        return np.argmax(y, axis=1).astype(np.int64)

    raise ValueError(f"Formato de y no soportado: shape={y.shape}")


def _build_optimizer_and_scheduler(model, compile_args):
    optimizer_name = compile_args["optimizer_name"].lower()
    optimizer_params = deepcopy(compile_args["optimizer_params"])

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    scheduler_name = compile_args.get("scheduler_name", None)
    scheduler = None

    if scheduler_name is not None:
        scheduler_name = scheduler_name.lower()
        scheduler_params = deepcopy(compile_args.get("scheduler_params", {}))

        if scheduler_name == "reducelronplateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, **scheduler_params
            )
        else:
            raise ValueError(f"Scheduler no soportado: {scheduler_name}")

    return optimizer, scheduler


def _run_epoch_pytorch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)  # (B, 2)

        optimizer.zero_grad()

        outputs = model(xb)  # dict con out_activation y kernel_entropy
        loss = loss_fn(outputs, yb)

        loss.backward()
        optimizer.step()

        # para imitar kernel_constraint=max_norm(norm_rate) del original
        apply_max_norm_to_linear(model.classifier, max_value=model.norm_rate)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

    return running_loss / max(n_samples, 1)


@torch.no_grad()
def _evaluate_pytorch_tgarnet(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    n_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)  # (B, 2)

        outputs = model(xb)
        probs_2c = outputs["out_activation"]            # (B, 2)
        loss = loss_fn(outputs, yb)

        probs_pos = probs_2c[:, 1]                      # prob clase ADHD/TDAH
        preds = torch.argmax(probs_2c, dim=1)
        y_true = torch.argmax(yb, dim=1)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

        all_probs.append(probs_pos.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y_true.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs).astype(np.float32)
    y_pred = np.concatenate(all_preds).astype(np.int64)
    y_true = np.concatenate(all_true).astype(np.int64)

    mean_loss = running_loss / max(n_samples, 1)

    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_value = np.nan

    metrics = {
        "loss": mean_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    return metrics


def train_L24O_cv(
    model_name,
    X,
    y,
    sbjs,
    folds,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    study_name="study_tgarnet_tdah",
    trial_number=0,
    trial_cache_root="optuna_resume_cache",
    trial_cache_key=None,
    cache_model_format="weights",
    force_retrain=False,
    evaluate_test=True,
):
    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    saved_model_paths = {}

    model_name = model_name.lower()
    if model_name != "tgarnet":
        raise ValueError("Esta versión solo soporta 'TGARNet'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    trial_subdir = trial_cache_key if trial_cache_key is not None else f"trial_{trial_number:04d}"

    trial_dir = os.path.join(
        trial_cache_root,
        study_name,
        trial_subdir
    )
    os.makedirs(trial_dir, exist_ok=True)

    for fold, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        fold_dir = os.path.join(trial_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        checkpoint_path = os.path.join(fold_dir, "resume_checkpoint.pt")
        best_state_path = os.path.join(fold_dir, "best_state.pt")
        fold_info_path = os.path.join(fold_dir, "fold_result.pkl")
        history_path = os.path.join(fold_dir, "history.pkl")

        if cache_model_format == "weights":
            cached_model_path = os.path.join(fold_dir, "final_state_dict.pt")
        elif cache_model_format == "full":
            cached_model_path = os.path.join(fold_dir, "final_model.pt")
        else:
            raise ValueError("cache_model_format debe ser 'weights' o 'full'")

        if force_retrain:
            for p in [checkpoint_path, best_state_path, fold_info_path, history_path, cached_model_path]:
                if os.path.exists(p):
                    os.remove(p)

        if (not force_retrain) and os.path.exists(fold_info_path) and os.path.exists(cached_model_path):
            with open(fold_info_path, "rb") as f:
                fold_info = pickle.load(f)

            cached_eval_mode = fold_info.get("evaluate_test", None)

            if cached_eval_mode == evaluate_test:
                if evaluate_test:
                    all_fold_metrics.append(fold_info["fold_metrics"])
                all_fold_val_metrics.append(fold_info["fold_val_metrics"])
                total_histories.append(fold_info["history"])
                saved_model_paths[fold] = cached_model_path
                print(f"[Trial {trial_number}] Fold {fold}: reutilizado desde disco.")
                continue
            else:
                print(
                    f"[Trial {trial_number}] Fold {fold}: caché incompatible "
                    f"(cached evaluate_test={cached_eval_mode}, actual={evaluate_test}). Reentrenando."
                )

        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        val_idx = [i for i, sbj in enumerate(sbjs) if sbj in val_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        y_train = ensure_onehot_labels(y_train)
        y_val = ensure_onehot_labels(y_val)
        if evaluate_test:
            y_test = ensure_onehot_labels(y_test)

        set_seed(seed + fold)

        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold

        model = build_eeg_model(model_name=model_name, **model_args_local).to(device)

        compile_cfg_local = deepcopy(compile_cfg)
        compile_args_local, callbacks = build_compile_config(
            model_name=model_name,
            Chans=model_args["Chans"],
            total_epochs=epochs,
            **compile_cfg_local
        )

        loss_fn = compile_args_local["loss_fn"]
        optimizer, scheduler = _build_optimizer_and_scheduler(model, compile_args_local)
        loss_weights = compile_args_local.get("loss_weights", None)

        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )
        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        g = torch.Generator()
        g.manual_seed(seed + fold)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
            generator=g,
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None
        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        start_epoch = 0
        best_val_loss = np.inf
        patience_counter = 0
        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
            "lambda_kernel_entropy": [],
        }

        if (not force_retrain) and os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

            if scheduler is not None and ckpt.get("scheduler_state_dict", None) is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt["best_val_loss"]
            patience_counter = ckpt["patience_counter"]
            history = ckpt["history"]

            if loss_weights is not None and "loss_weights" in ckpt:
                loss_weights.update(ckpt["loss_weights"])

            print(f"[Trial {trial_number}] Fold {fold}: reanudando desde epoch {start_epoch}.")

        for epoch in range(start_epoch, epochs):
            for cb in callbacks:
                if hasattr(cb, "on_epoch_begin") and loss_weights is not None:
                    cb.on_epoch_begin(epoch, loss_weights)

            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_tgarnet(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]
            lambda_curr = loss_weights["kernel_entropy"] if loss_weights is not None else np.nan

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)
            history["lambda_kernel_entropy"].append(lambda_curr)

            improved = val_loss < (best_val_loss - 1e-4)
            if improved:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), best_state_path)
            else:
                patience_counter += 1

            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "best_val_loss": best_val_loss,
                "patience_counter": patience_counter,
                "history": history,
                "loss_weights": loss_weights,
            }
            torch.save(checkpoint, checkpoint_path)

            with open(history_path, "wb") as f:
                pickle.dump(history, f)

            if patience_counter >= 25:
                break

        if os.path.exists(best_state_path):
            best_state = torch.load(best_state_path, map_location=device)
            model.load_state_dict(best_state)

        val_eval = _evaluate_pytorch_tgarnet(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        if evaluate_test:
            test_eval = _evaluate_pytorch_tgarnet(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }
        else:
            fold_metrics = None

        if cache_model_format == "weights":
            torch.save(model.state_dict(), cached_model_path)
        else:
            torch.save(model, cached_model_path)

        fold_info = {
            "fold_metrics": fold_metrics,
            "fold_val_metrics": fold_val_metrics,
            "history": history,
            "evaluate_test": evaluate_test,
        }

        with open(fold_info_path, "wb") as f:
            pickle.dump(fold_info, f)

        if evaluate_test:
            all_fold_metrics.append(fold_metrics)
        all_fold_val_metrics.append(fold_val_metrics)
        total_histories.append(history)
        saved_model_paths[fold] = cached_model_path

        print(f"[Trial {trial_number}] Fold {fold}: entrenado y guardado.")

    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}
        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = np.nanmean(vals)
            mean_metrics[f"std_{key}"] = np.nanstd(vals)

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    mean_val_metrics = {}
    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = np.nanmean(vals)
        mean_val_metrics[f"std_{key}"] = np.nanstd(vals)

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]

    return {
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "saved_model_paths": saved_model_paths,
        "trial_dir": trial_dir,
    }


def train_npz_folds_cv(
    npz_path,
    model_name,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    evaluate_test=True,
):
    data = np.load(npz_path)

    X = data["X"].astype(np.float32)
    y = ensure_onehot_labels(data["y"])

    fold_numbers = sorted({
        int(key.split("_")[1])
        for key in data.files
        if key.startswith("fold_") and key.endswith("_train_idx")
    })

    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    models = {}

    model_name = model_name.lower()
    if model_name != "tgarnet":
        raise ValueError("Esta versión solo soporta 'TGARNet'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold_pos, fold_num in enumerate(fold_numbers):
        train_idx = data[f"fold_{fold_num}_train_idx"]
        test_idx = data[f"fold_{fold_num}_test_idx"]
        val_idx = data[f"fold_{fold_num}_val_idx"]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        y_train = ensure_onehot_labels(y_train)
        y_val = ensure_onehot_labels(y_val)
        if evaluate_test:
            y_test = ensure_onehot_labels(y_test)

        set_seed(seed + fold_pos)

        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold_pos

        model = build_eeg_model(model_name=model_name, **model_args_local).to(device)

        compile_cfg_local = deepcopy(compile_cfg)
        compile_args_local, callbacks = build_compile_config(
            model_name=model_name,
            Chans=model_args["Chans"],
            total_epochs=epochs,
            **compile_cfg_local
        )

        loss_fn = compile_args_local["loss_fn"]
        optimizer, scheduler = _build_optimizer_and_scheduler(model, compile_args_local)
        loss_weights = compile_args_local.get("loss_weights", None)

        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )
        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None
        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        best_val_loss = np.inf
        best_state = deepcopy(model.state_dict())
        patience_counter = 0

        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
            "lambda_kernel_entropy": [],
        }

        for epoch in range(epochs):
            for cb in callbacks:
                if hasattr(cb, "on_epoch_begin") and loss_weights is not None:
                    cb.on_epoch_begin(epoch, loss_weights)

            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_tgarnet(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]
            lambda_curr = loss_weights["kernel_entropy"] if loss_weights is not None else np.nan

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)
            history["lambda_kernel_entropy"].append(lambda_curr)

            improved = val_loss < (best_val_loss - 1e-4)
            if improved:
                best_val_loss = val_loss
                best_state = deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= 25:
                break

        model.load_state_dict(best_state)
        total_histories.append(history)

        val_eval = _evaluate_pytorch_tgarnet(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        all_fold_val_metrics.append(fold_val_metrics)

        if evaluate_test:
            test_eval = _evaluate_pytorch_tgarnet(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }

            all_fold_metrics.append(fold_metrics)

        models[fold_num] = model

    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}
        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = np.nanmean(vals)
            mean_metrics[f"std_{key}"] = np.nanstd(vals)

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    mean_val_metrics = {}
    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = np.nanmean(vals)
        mean_val_metrics[f"std_{key}"] = np.nanstd(vals)

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]

    return {
        "X": X,
        "y": y,
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "models": models,
    }

In [23]:
def make_trial_cache_key(model_args, compile_cfg):
    payload = {
        "model_args": model_args,
        "compile_cfg": compile_cfg,
    }
    text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:16]


def make_objective(
    model_name,
    data_mode,
    base_model_args,
    base_compile_args,
    epochs=100,
    batch_size=16,
    seed=42,
    X=None,
    y=None,
    sbjs=None,
    folds=None,
    best_model_dir="best_models",
    save_format="weights",
    direction="maximize",
    resume_root="optuna_resume_cache",
    study_name="study_tgarnet_tdah",
    force_retrain=False,
):
    def objective(trial):
        model_args = suggest_model_args(
            trial=trial,
            model_name=model_name,
            base_model_args=base_model_args
        )

        compile_cfg = suggest_compile_args(
            trial=trial,
            model_name=model_name,
            base_compile_args=base_compile_args
        )

        trial_cache_key = make_trial_cache_key(model_args, compile_cfg)

        if data_mode != "subject_folds":
            raise ValueError("Esta versión está preparada solo para 'subject_folds' (TDAH).")

        results = train_L24O_cv(
            model_name=model_name,
            X=X,
            y=y,
            sbjs=sbjs,
            folds=folds,
            model_args=model_args,
            compile_cfg=compile_cfg,
            epochs=epochs,
            batch_size=batch_size,
            seed=seed,
            study_name=study_name,
            trial_number=trial.number,
            trial_cache_root=resume_root,
            trial_cache_key=trial_cache_key,
            cache_model_format=save_format,
            force_retrain=force_retrain,
            evaluate_test=False,
        )

        objective_value = float(results["mean_val_accuracy"])

        try:
            prev_best = trial.study.best_value
            has_prev_best = True
        except ValueError:
            prev_best = None
            has_prev_best = False

        if direction == "maximize":
            is_new_best = (not has_prev_best) or (objective_value > prev_best)
        elif direction == "minimize":
            is_new_best = (not has_prev_best) or (objective_value < prev_best)
        else:
            raise ValueError("direction debe ser 'maximize' o 'minimize'")

        if is_new_best:
            if os.path.exists(best_model_dir):
                shutil.rmtree(best_model_dir)

            os.makedirs(best_model_dir, exist_ok=True)

            for fold_id, cached_model_path in results["saved_model_paths"].items():
                if save_format == "weights":
                    dst_path = os.path.join(
                        best_model_dir,
                        f"{model_name}_BESTTRIAL_fold{fold_id}.state_dict.pt"
                    )
                elif save_format == "full":
                    dst_path = os.path.join(
                        best_model_dir,
                        f"{model_name}_BESTTRIAL_fold{fold_id}.model.pt"
                    )
                else:
                    raise ValueError("save_format debe ser 'weights' o 'full'")

                shutil.copy2(cached_model_path, dst_path)

            trial.set_user_attr("saved_as_best", True)
            trial.set_user_attr("best_model_dir", best_model_dir)
        else:
            trial.set_user_attr("saved_as_best", False)

        trial.set_user_attr("model_args", model_args)
        trial.set_user_attr("compile_cfg", compile_cfg)
        trial.set_user_attr("trial_cache_key", trial_cache_key)
        trial.set_user_attr("mean_val_accuracy", objective_value)
        trial.set_user_attr("trial_dir", results["trial_dir"])

        return objective_value

    return objective

In [24]:
def get_or_create_study_local(
    study_name,
    journal_file,
    direction="maximize",
    seed=42,
):
    """
    Crea o reutiliza un estudio Optuna con JournalStorage local.

    Parámetros
    ----------
    study_name : str
        Nombre del estudio.
    journal_file : str
        Ruta al archivo .journal donde se guardará el historial.
    direction : str
        'maximize' o 'minimize'.
    seed : int
        Semilla para el sampler.

    Retorna
    -------
    optuna.study.Study
    """
    journal_dir = os.path.dirname(os.path.abspath(journal_file))
    if journal_dir and not os.path.exists(journal_dir):
        os.makedirs(journal_dir, exist_ok=True)

    storage = JournalStorage(JournalFileBackend(journal_file))

    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction=direction,
        load_if_exists=True,
        sampler=optuna.samplers.GPSampler(
            seed=seed,
            deterministic_objective=True,
        ),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )

    return study

In [25]:
def run_optuna_experiment(
    model_name,
    data_mode,
    study_name,
    journal_file,
    base_model_args,
    base_compile_args,
    epochs=100,
    batch_size=16,
    seed=42,
    X=None,
    y=None,
    sbjs=None,
    folds=None,
    n_trials_total=20,
    best_model_dir="best_models",
    save_format="weights",
    resume_root="optuna_resume_cache",
    force_retrain=False,
):
    objective = make_objective(
        model_name=model_name,
        data_mode=data_mode,
        base_model_args=base_model_args,
        base_compile_args=base_compile_args,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        best_model_dir=best_model_dir,
        save_format=save_format,
        direction="maximize",
        resume_root=resume_root,
        study_name=study_name,
        force_retrain=force_retrain,
    )

    study = get_or_create_study_local(
        study_name=study_name,
        journal_file=journal_file,
        direction="maximize",
        seed=seed,
    )

    completed_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    unfinished_trials = [
        t for t in study.trials
        if t.state in (optuna.trial.TrialState.RUNNING, optuna.trial.TrialState.WAITING)
    ]

    requeued_count = 0
    seen_param_signatures = set()

    for t in unfinished_trials:
        if len(t.params) == 0:
            continue

        param_signature = tuple(sorted(t.params.items()))
        if param_signature in seen_param_signatures:
            continue

        study.enqueue_trial(t.params, skip_if_exists=False)
        seen_param_signatures.add(param_signature)
        requeued_count += 1

    remaining_completed_needed = max(0, n_trials_total - len(completed_trials))

    print(f"Study name                : {study_name}")
    print(f"Trials completos          : {len(completed_trials)}")
    print(f"Trials incompletos        : {len(unfinished_trials)}")
    print(f"Reencolados               : {requeued_count}")
    print(f"Meta total (completos)    : {n_trials_total}")
    print(f"Faltantes por completar   : {remaining_completed_needed}")

    if remaining_completed_needed > 0:
        study.optimize(
            objective,
            n_trials=remaining_completed_needed,
        )
    else:
        print("El estudio ya alcanzó o superó el número objetivo de trials completos.")

    try:
        print("Best value:", study.best_value)
        print("Best params:", study.best_params)
    except ValueError:
        print("Aún no hay trials completos con best_value disponible.")

    return study

In [26]:
import pickle
import os
import hashlib
import json
import optuna
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from optuna.exceptions import ExperimentalWarning
    warnings.filterwarnings("ignore", category=ExperimentalWarning)
except Exception:
    pass

model_name = "TGARNet"
db = "TDAH"
seed = 42
n_trials_total = 20

study_name = f"study_{model_name}_{db}"

# ==============================
# RUTA DIRECTA DE GUARDADO
# ==============================
SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah"
os.makedirs(SAVE_ROOT, exist_ok=True)

# Carpeta para mejores modelos
BEST_MODEL_DIR = os.path.join(SAVE_ROOT, f"{model_name}_best_models")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Carpeta para caché/reanudación
RESUME_ROOT = os.path.join(SAVE_ROOT, "optuna_resume_cache")
os.makedirs(RESUME_ROOT, exist_ok=True)

# Archivo journal de Optuna
JOURNAL_FILE = os.path.join(SAVE_ROOT, f"{study_name}.journal")

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

# get_segmented_data() ya debe devolver y en one-hot para T-GARNet
X, y, sbjs, _ = get_segmented_data()

study = run_optuna_experiment(
    model_name=model_name,
    data_mode="subject_folds",
    study_name=study_name,
    journal_file=JOURNAL_FILE,
    base_model_args={
        "nb_classes": 2,
        "Chans": 19,
        "Samples": 512,
        "norm_rate": 0.25,
        "alpha": 2,
        "dropoutRate": 0.3,
    },
    base_compile_args={},
    epochs=100,
    batch_size=16,
    seed=seed,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    n_trials_total=n_trials_total,
    best_model_dir=BEST_MODEL_DIR,
    save_format="weights",
    resume_root=RESUME_ROOT,
    force_retrain=True,
)

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINALIZADO")
print("=" * 80)
print(f"Carpeta principal       : {SAVE_ROOT}")
print(f"Study name              : {study_name}")
print(f"Journal de Optuna       : {JOURNAL_FILE}")
print(f"Mejores modelos         : {BEST_MODEL_DIR}")
print(f"Caché de reanudación    : {RESUME_ROOT}")

if study is not None:
    try:
        print(f"Mejor valor del estudio : {study.best_value}")
        print("Mejores hiperparámetros:")
        for k, v in study.best_trial.params.items():
            print(f"  {k}: {v}")
    except ValueError:
        print("Aún no hay trials completos válidos.")

[I 2026-04-30 16:52:55,547] Using an existing study with name 'study_TGARNet_TDAH' instead of creating a new one.


Study name                : study_TGARNet_TDAH
Trials completos          : 20
Trials incompletos        : 0
Reencolados               : 0
Meta total (completos)    : 20
Faltantes por completar   : 0
El estudio ya alcanzó o superó el número objetivo de trials completos.
Best value: 0.8673441579415571
Best params: {'filters': 2, 'kernel_sigma': 20.0, 'num_heads': 3, 'intermediate_dim': 16, 'learning_rate': 0.0001, 'schedule_delta': 1.0}

ENTRENAMIENTO FINALIZADO
Carpeta principal       : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah
Study name              : study_TGARNet_TDAH
Journal de Optuna       : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah/study_TGARNet_TDAH.journal
Mejores modelos         : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah/TGARNet_best_models
Caché de reanudación    : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah/optuna_resume_cache
Mejor valor del estudio : 0.8673441579415571
Me

In [27]:
import os
import json
import pickle
import numpy as np

# ==============================
# 1) Cargar estudio
# ==============================
study = get_or_create_study_local(
    study_name=f"study_{model_name}_{db}",
    journal_file=JOURNAL_FILE,
    direction="maximize",
    seed=seed,
)

print("Best value (val):", study.best_value)
print("Best trial number:", study.best_trial.number)
print("Best params:", study.best_trial.params)
print("Best trial user_attrs:", study.best_trial.user_attrs)

# ==============================
# 2) Reconstruir args del mejor trial
# ==============================
best_trial = study.best_trial

model_args = suggest_model_args(
    trial=best_trial,
    model_name=model_name,
    base_model_args={
        "nb_classes": 2,
        "Chans": 19,
        "Samples": 512,
        "alpha": 2,
    }
)

compile_cfg = suggest_compile_args(
    trial=best_trial,
    model_name=model_name,
    base_compile_args={}
)

# importante: verificar que exista trial_cache_key
if "trial_cache_key" not in best_trial.user_attrs:
    raise KeyError(
        "best_trial.user_attrs no contiene 'trial_cache_key'. "
        "Así train_L24O_cv podría no encontrar la caché correcta."
    )

trial_cache_key = best_trial.user_attrs["trial_cache_key"]

# ==============================
# 3) Recuperar resultados del mejor trial
#    (debería reutilizar caché)
# ==============================
results = train_L24O_cv(
    model_name=model_name,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    model_args=model_args,
    compile_cfg=compile_cfg,
    epochs=100,
    batch_size=16,
    seed=seed,
    study_name=f"study_{model_name}_{db}",
    trial_number=best_trial.number,
    trial_cache_root=RESUME_ROOT,
    trial_cache_key=trial_cache_key,
    cache_model_format="weights",
    force_retrain=False,   # si tu función lo soporta
)

# ==============================
# 4) Imprimir accuracy test y std
# ==============================
print("\n" + "=" * 80)
print("MEJOR TRIAL - RESULTADOS EN TEST")
print("=" * 80)

print(f"Accuracy medio en test : {results['mean_accuracy']:.4f}")

if "mean_metrics" in results and "std_accuracy" in results["mean_metrics"]:
    print(f"STD accuracy en test   : {results['mean_metrics']['std_accuracy']:.4f}")

print("\nAccuracy por fold:")
for i, fm in enumerate(results["fold_metrics"]):
    print(f"  Fold {i}: {fm['accuracy']:.4f}")

print("\nResumen test:")
for k, v in results["mean_metrics"].items():
    print(f"  {k}: {v:.4f}")

[I 2026-04-30 16:52:55,569] Using an existing study with name 'study_TGARNet_TDAH' instead of creating a new one.


Best value (val): 0.8673441579415571
Best trial number: 19
Best params: {'filters': 2, 'kernel_sigma': 20.0, 'num_heads': 3, 'intermediate_dim': 16, 'learning_rate': 0.0001, 'schedule_delta': 1.0}
Best trial user_attrs: {'saved_as_best': True, 'best_model_dir': '/home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah/TGARNet_best_models', 'model_args': {'nb_classes': 2, 'Chans': 19, 'Samples': 512, 'norm_rate': 0.25, 'alpha': 2, 'dropoutRate': 0.3, 'filters': 2, 'kernel_sigma': 20.0, 'num_heads': 3, 'intermediate_dim': 16}, 'compile_cfg': {'learning_rate': 0.0001, 'schedule_delta': 1.0}, 'trial_cache_key': '648e04b9bd8cbf75', 'mean_val_accuracy': 0.8673441579415571, 'trial_dir': '/home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah/optuna_resume_cache/study_TGARNet_TDAH/648e04b9bd8cbf75'}
[Trial 19] Fold 0: reutilizado desde disco.
[Trial 19] Fold 1: reutilizado desde disco.
[Trial 19] Fold 2: reutilizado desde disco.
[Trial 19] Fold 3: reutilizado desde disc

In [28]:
import os
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================================================
# 1) CONFIGURACIÓN GENERAL
# =========================================================
model_name = "TGARNet"
db = "TDAH"

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_tgarnet_tdah_repeated"
os.makedirs(SAVE_ROOT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_ROOT, "repeated_test_results.csv")
RESULTS_JSON = os.path.join(SAVE_ROOT, "repeated_test_summary.json")

# 10 repeticiones -> cada seed corre los 5 folds
seeds = list(range(10))

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# 2) MEJORES HIPERPARÁMETROS DEL NUEVO ESTUDIO TGARNet
#    Best trial = 19
# =========================================================
best_params = {
    "filters": 2,
    "kernel_sigma": 20.0,
    "num_heads": 3,
    "intermediate_dim": 16,
    "learning_rate": 0.0001,
    "schedule_delta": 1.0,
}

# =========================================================
# 3) FUNCIONES AUXILIARES
# =========================================================
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def summarize_metric(values):
    arr = np.array(values, dtype=float)
    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "n": int(len(arr)),
    }

# =========================================================
# 4) CARGAR DATOS Y FOLDS
# =========================================================
with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data()

# =========================================================
# 5) ARMAR model_args Y compile_cfg
#    usando best_params fijos del mejor trial TGARNet
# =========================================================
model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
    "norm_rate": 0.25,
    "alpha": 2,
    "num_heads": best_params["num_heads"],
    "intermediate_dim": best_params["intermediate_dim"],
    "filters": best_params["filters"],
    "dropoutRate": 0.3,
    "kernel_sigma": best_params["kernel_sigma"],
}

compile_cfg = {
    "learning_rate": best_params["learning_rate"],
    "schedule_delta": best_params["schedule_delta"],
}

print("=" * 90)
print("CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - TGARNet")
print("=" * 90)
print("model_args:")
for k, v in model_args.items():
    print(f"  {k}: {v}")

print("\ncompile_cfg:")
for k, v in compile_cfg.items():
    print(f"  {k}: {v}")

# =========================================================
# 6) REPETICIONES
#    train_L24O_cv corre todos los folds.
#    Cada seed genera métricas para los 5 folds.
# =========================================================
all_rows = []
all_run_summaries = []

for rep_id, seed in enumerate(seeds):
    print("\n" + "=" * 90)
    print(f"REPETICIÓN {rep_id+1}/{len(seeds)} | seed={seed}")
    print("=" * 90)

    set_all_seeds(seed)

    rep_cache_root = os.path.join(SAVE_ROOT, f"repeat_seed_{seed}")
    os.makedirs(rep_cache_root, exist_ok=True)

    results = train_L24O_cv(
        model_name=model_name,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        model_args=model_args,
        compile_cfg=compile_cfg,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        study_name=f"fixed_best_params_{model_name}_{db}",
        trial_number=rep_id,
        trial_cache_root=rep_cache_root,
        trial_cache_key=f"fixedparams_tgarnet_seed_{seed}",
        cache_model_format=save_format,
        force_retrain=force_retrain,
        evaluate_test=True,
    )

    fold_metrics = results["fold_metrics"]

    for fold_id, fm in enumerate(fold_metrics):
        row = {
            "seed": seed,
            "repeat_id": rep_id,
            "fold": fold_id,
            "accuracy": float(fm["accuracy"]),
            "recall": float(fm["recall"]),
            "precision": float(fm["precision"]),
            "kappa": float(fm["kappa"]),
            "auc": float(fm["auc"]),
        }
        all_rows.append(row)

    run_summary = {
        "seed": seed,
        "repeat_id": rep_id,
        "mean_accuracy_across_5_folds": float(results["mean_accuracy"]),
        "mean_metrics": {
            k: float(v) for k, v in results["mean_metrics"].items()
        },
    }
    all_run_summaries.append(run_summary)

# =========================================================
# 7) GUARDAR RESULTADOS CRUDOS
# =========================================================
df = pd.DataFrame(all_rows)
df.to_csv(RESULTS_CSV, index=False)

print("\nArchivo CSV guardado en:")
print(RESULTS_CSV)

# =========================================================
# 8) RESUMEN GLOBAL SOBRE LAS 50 CORRIDAS
#    (5 folds x 10 seeds)
# =========================================================
summary_global = {
    "accuracy": summarize_metric(df["accuracy"].tolist()),
    "recall": summarize_metric(df["recall"].tolist()),
    "precision": summarize_metric(df["precision"].tolist()),
    "kappa": summarize_metric(df["kappa"].tolist()),
    "auc": summarize_metric(df["auc"].tolist()),
}

# =========================================================
# 9) RESUMEN POR FOLD
# =========================================================
summary_by_fold = {}
for fold_id in sorted(df["fold"].unique()):
    dff = df[df["fold"] == fold_id]
    summary_by_fold[f"fold_{fold_id}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }

# =========================================================
# 10) RESUMEN POR SEED
# =========================================================
summary_by_seed = {}
for seed_val in sorted(df["seed"].unique()):
    dff = df[df["seed"] == seed_val]
    summary_by_seed[f"seed_{seed_val}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }

# =========================================================
# 11) GUARDAR JSON FINAL
# =========================================================
final_summary = {
    "model_name": model_name,
    "database": db,
    "best_trial": 19,
    "best_params": best_params,
    "n_folds": 5,
    "n_repetitions": len(seeds),
    "total_test_runs": int(len(df)),
    "global_summary_over_50_runs": summary_global,
    "summary_by_fold": summary_by_fold,
    "summary_by_seed": summary_by_seed,
    "run_summaries": all_run_summaries,
    "csv_path": RESULTS_CSV,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("\nArchivo JSON guardado en:")
print(RESULTS_JSON)

# =========================================================
# 12) IMPRIMIR RESULTADOS FINALES
# =========================================================
print("\n" + "=" * 90)
print("RESULTADOS FINALES - 5 FOLDS x 10 REPETICIONES = 50 TESTS")
print("=" * 90)

print(f"Accuracy  : {summary_global['accuracy']['mean']:.4f} ± {summary_global['accuracy']['std_sample']:.4f}")
print(f"Recall    : {summary_global['recall']['mean']:.4f} ± {summary_global['recall']['std_sample']:.4f}")
print(f"Precision : {summary_global['precision']['mean']:.4f} ± {summary_global['precision']['std_sample']:.4f}")
print(f"Kappa     : {summary_global['kappa']['mean']:.4f} ± {summary_global['kappa']['std_sample']:.4f}")
print(f"AUC       : {summary_global['auc']['mean']:.4f} ± {summary_global['auc']['std_sample']:.4f}")

print("\nResumen por fold:")
for fold_name, vals in summary_by_fold.items():
    print(f"{fold_name}: Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f}")

print("\nResumen por seed:")
for seed_name, vals in summary_by_seed.items():
    print(f"{seed_name}: Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f}")

CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - TGARNet
model_args:
  nb_classes: 2
  Chans: 19
  Samples: 512
  norm_rate: 0.25
  alpha: 2
  num_heads: 3
  intermediate_dim: 16
  filters: 2
  dropoutRate: 0.3
  kernel_sigma: 20.0

compile_cfg:
  learning_rate: 0.0001
  schedule_delta: 1.0

REPETICIÓN 1/10 | seed=0
[Trial 0] Fold 0: entrenado y guardado.
[Trial 0] Fold 1: entrenado y guardado.
[Trial 0] Fold 2: entrenado y guardado.
[Trial 0] Fold 3: entrenado y guardado.
[Trial 0] Fold 4: entrenado y guardado.

REPETICIÓN 2/10 | seed=1
[Trial 1] Fold 0: entrenado y guardado.
[Trial 1] Fold 1: entrenado y guardado.
[Trial 1] Fold 2: entrenado y guardado.
[Trial 1] Fold 3: entrenado y guardado.
[Trial 1] Fold 4: entrenado y guardado.

REPETICIÓN 3/10 | seed=2
[Trial 2] Fold 0: entrenado y guardado.
[Trial 2] Fold 1: entrenado y guardado.
[Trial 2] Fold 2: entrenado y guardado.
[Trial 2] Fold 3: entrenado y guardado.
[Trial 2] Fold 4: entrenado y guardado.

REPETICIÓN 4/10 | seed=3
[Trial